# 04 - Resultados y Perfiles de Negocio (RFM)
Traduccion de los clusters a patrones de negocio (Clientes VIP, en
Riesgo de Abandono, Esporadicos de Bajo Ticket, Regulares) para que la
Licenciada Claudia disene estrategias de marketing directo.

## Importar dependencias

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import squarify
import sys
import os

sys.path.append("../src")
from segmentacion import etiquetar_segmentos, resumen_clusters
from conexion import guardar_csv

sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (12, 5)
os.makedirs("../outputs/graficos", exist_ok=True)

## Cargar los datos clusterizados

In [ ]:
df = pd.read_csv("../data/clientes_clusterizados.csv")
print(f"Datos cargados: {df.shape}")
df.head()

## Etiquetar los segmentos de negocio

In [ ]:
df, segmentos_map = etiquetar_segmentos(df)
print("\nMapa de Segmentos (cluster -> patron):")
for cluster_id, segmento in segmentos_map.items():
    print(f"Cluster {cluster_id}: {segmento}")

## Resumen estadistico por cluster

In [ ]:
resumen = resumen_clusters(df)
guardar_csv(resumen.reset_index(), "../data/resumen_clusters.csv")
print("Archivo resumen_clusters.csv guardado")

## Perfil RFM por segmento (radar de puntajes)
Compara de un vistazo el ADN de cada segmento en sus tres puntajes (1-5).
Cuanto mas grande el poligono, mejor cliente.

In [ ]:
segmentos = sorted(df['Segmento'].unique())
metricas = ['Puntaje_Proximidad', 'Puntaje_Frecuencia', 'Puntaje_ValorMonetario']
ejes = ['Proximidad', 'Frecuencia', 'Valor Monetario']
perfil = df.groupby('Segmento')[metricas].mean().reindex(segmentos)

angulos = np.linspace(0, 2 * np.pi, len(metricas), endpoint=False).tolist()
angulos += angulos[:1]  # cerrar el poligono

fig, ax = plt.subplots(figsize=(8, 8), subplot_kw=dict(polar=True))
colores = sns.color_palette('Set2', len(segmentos))
for seg, color in zip(segmentos, colores):
    valores = perfil.loc[seg].tolist()
    valores += valores[:1]
    ax.plot(angulos, valores, color=color, linewidth=2, label=seg)
    ax.fill(angulos, valores, color=color, alpha=0.15)
ax.set_xticks(angulos[:-1])
ax.set_xticklabels(ejes, fontsize=11)
ax.set_ylim(0, 5)
ax.set_yticks([1, 2, 3, 4, 5])
ax.set_title('Perfil RFM por Segmento (radar de puntajes 1-5)', pad=20)
ax.legend(loc='upper right', bbox_to_anchor=(1.35, 1.10))
plt.tight_layout()
plt.savefig('../outputs/graficos/radar_rfm_segmentos.png', dpi=300, bbox_inches='tight')
plt.show()

## Mapa de segmentos RFM (burbujas)
Vista ejecutiva de la cartera: eje X = frecuencia, eje Y = proximidad
(invertido, los mas recientes arriba) y el **tamano de la burbuja = valor
monetario medio**. Permite ubicar cada segmento como un cuadrante de accion.

In [ ]:
agg = df.groupby('Segmento').agg(
    Frecuencia=('Frecuencia', 'mean'),
    Proximidad=('Proximidad', 'mean'),
    ValorMonetario=('ValorMonetario', 'mean'),
    Clientes=('CodigoCliente', 'count'),
).reindex(segmentos)

fig, ax = plt.subplots(figsize=(11, 7))
colores = dict(zip(segmentos, sns.color_palette('Set2', len(segmentos))))
vmax = agg['ValorMonetario'].max() if agg['ValorMonetario'].max() > 0 else 1
for seg in segmentos:
    r = agg.loc[seg]
    ax.scatter(r['Frecuencia'], r['Proximidad'],
               s=np.sqrt(r['ValorMonetario'] / vmax) * 5500 + 700,
               color=colores[seg], alpha=0.55, edgecolors='black', linewidth=1.3)
    ax.annotate(f"{seg}\n{int(r['Clientes']):,} clientes\nQ{r['ValorMonetario']:,.0f}",
                (r['Frecuencia'], r['Proximidad']),
                ha='center', va='center', fontsize=8, weight='bold')
ax.invert_yaxis()  # menor proximidad (mas reciente) hacia arriba
ax.set_xlabel('Frecuencia media (visitas)')
ax.set_ylabel('Proximidad media (dias desde la ultima visita)')
ax.set_title('Mapa de Segmentos RFM  ·  tamano de burbuja = valor monetario medio')
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('../outputs/graficos/burbujas_rfm_segmentos.png', dpi=300, bbox_inches='tight')
plt.show()

## Participacion de cada segmento (treemap)
El area de cada bloque es proporcional al **numero de clientes** del segmento;
ideal para comunicar a direccion cuanto pesa cada grupo en la cartera.

In [ ]:
agg = df.groupby('Segmento').agg(
    Clientes=('CodigoCliente', 'count'),
    ValorMonetario=('ValorMonetario', 'mean'),
).reindex(segmentos)
total = agg['Clientes'].sum()
etiquetas = [f"{seg}\n{c:,} clientes ({c / total * 100:.1f}%)\nticket medio Q{v:,.0f}"
             for seg, c, v in zip(agg.index, agg['Clientes'], agg['ValorMonetario'])]

fig, ax = plt.subplots(figsize=(12, 7))
colores = sns.color_palette('Set2', len(segmentos))
squarify.plot(sizes=agg['Clientes'], label=etiquetas, color=colores, alpha=0.85,
              pad=True, text_kwargs={'fontsize': 10, 'weight': 'bold'}, ax=ax)
ax.axis('off')
ax.set_title('Participacion de Clientes por Segmento (treemap)')
plt.tight_layout()
plt.savefig('../outputs/graficos/treemap_segmentos.png', dpi=300, bbox_inches='tight')
plt.show()

## Estadisticas y recomendacion de marketing por segmento

In [ ]:
recomendaciones = {
    'Clientes VIP': 'Programa de fidelidad, atencion preferencial y upselling de servicios premium.',
    'Clientes en Riesgo de Abandono': 'Campana de reactivacion: descuentos personalizados y recordatorios de mantenimiento.',
    'Clientes Esporadicos de Bajo Ticket': 'Promociones de bajo costo y paquetes de servicios para aumentar frecuencia.',
    'Clientes Regulares': 'Mantener comunicacion constante y ofertas estacionales para subir su valor.',
}
for seg in sorted(df['Segmento'].unique()):
    g = df[df['Segmento'] == seg]
    print(f"\n{'='*60}")
    print(f"Segmento: {seg}")
    print(f"{'='*60}")
    print(f"Total de clientes: {len(g)} ({len(g)/len(df)*100:.1f}%)")
    print(f"Proximidad promedio:     {g['Proximidad'].mean():,.1f} dias")
    print(f"Frecuencia promedio:     {g['Frecuencia'].mean():,.1f} visitas")
    print(f"ValorMonetario promedio: Q {g['ValorMonetario'].mean():,.2f}")
    print(f"Estrategia: {recomendaciones.get(seg, 'N/D')}")

## Exportar resultados finales

In [ ]:
columnas = ['CodigoCliente', 'NombreCliente', 'Proximidad', 'Frecuencia', 'ValorMonetario',
            'Puntaje_Proximidad', 'Puntaje_Frecuencia', 'Puntaje_ValorMonetario',
            'Puntaje_RFM', 'Cluster', 'Segmento']
df_export = df[[c for c in columnas if c in df.columns]].copy()
guardar_csv(df_export, "../data/clientes_rfm_segmentados.csv")
print(f"Total de registros exportados: {len(df_export)}")
df_export.head()